In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TABULAR DATASET

In [ ]:
df1=pd.read_csv("Tabular.csv")
df1.head()

In [ ]:
df1.info()

In [ ]:
df1.shape

In [ ]:
df1.describe()

In [ ]:
df1.isnull().sum()

In [ ]:
duplicates = df1[df1.duplicated()]
print(f"Number of duplicate rows: {duplicates.shape[0]}")

In [ ]:
# Convert 'Date' column to datetime objects
df1['Date'] = pd.to_datetime(df1['Date'], unit='D', origin='1899-12-30')

df1.head()

In [ ]:
df1['Date'].unique()

In [ ]:
price_correlations = df1.corr()['Price'].sort_values(ascending=False)

print("Feature Correlations with Price (Sorted):")
display(price_correlations)

In [ ]:
top_correlated_features = price_correlations.drop('Price').head(10).index.tolist()
display(df1[top_correlated_features].describe())

In [ ]:
display(df1[top_correlated_features].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get all numerical columns from the DataFrame
numerical_cols = df1.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols_filtered = [col for col in numerical_cols if col not in ['id', 'Date', 'Postal Code']]

num_cols = 4
num_rows = (len(numerical_cols_filtered) + num_cols - 1) // num_cols

plt.figure(figsize=(num_cols * 4, num_rows * 3))

for i, col in enumerate(numerical_cols_filtered):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(y=df1[col])
    plt.title(f'Box plot of {col}', fontsize=10)
    plt.ylabel('') # Remove y-label to avoid clutter
    plt.ticklabel_format(style='plain', axis='y') # Prevent scientific notation for y-axis

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(df1[numerical_cols_filtered].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

In [ ]:

df_cleaned = df1.copy()

outlier_cols = ['living area', 'lot area', 'Price', 'number of bedrooms', 'number of bathrooms', 'living_area_renov', 'lot_area_renov']

initial_rows = df_cleaned.shape[0]

for col in outlier_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR

    df_cleaned = df_cleaned[(df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)]

rows_removed = initial_rows - df_cleaned.shape[0]
print(f"Number of rows removed due to extreme outliers: {rows_removed}")
print(f"New DataFrame shape after outlier removal: {df_cleaned.shape}")

display(df_cleaned[outlier_cols].describe())

In [ ]:
numerical_cols_cleaned = df_cleaned.select_dtypes(include=['int64', 'float64']).columns.tolist()
skewness_values = df_cleaned[numerical_cols_cleaned].skew().sort_values(ascending=False)

print("Skewness before transformation (for df_cleaned):")
display(skewness_values.head(10))

skew_threshold = 0.75
for col in skewness_values[skewness_values > skew_threshold].index:
    if col not in ['id', 'Postal Code', 'Date'] and (df_cleaned[col] >= 0).all(): # Exclude identifier/date columns
        df_cleaned[col] = np.log1p(df_cleaned[col])
        print(f"Applied log1p transformation to '{col}'")
    elif not (df_cleaned[col] >= 0).all():
        print(f"Warning: '{col}' contains negative values, skipping log transformation.")

print("\nSkewness after transformation (for transformed columns in df_cleaned):")
display(df_cleaned[numerical_cols_cleaned].skew().sort_values(ascending=False).head(10))

In [ ]:

display(df_cleaned[top_correlated_features].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

transformed_cols_for_plot = [
    'Price',
    'lot area',
    'living area',
    'waterfront present',
    'number of views',
    'Area of the basement'
]

plt.figure(figsize=(15, 12))

for i, col in enumerate(transformed_cols_for_plot):
    # Plot original distribution (from df1)
    plt.subplot(len(transformed_cols_for_plot), 2, 2*i + 1)
    sns.histplot(df1[col], kde=True)
    plt.title(f'Original Distribution: {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

    # Plot transformed distribution (from df_cleaned)
    plt.subplot(len(transformed_cols_for_plot), 2, 2*i + 2)
    sns.histplot(df_cleaned[col], kde=True, color='orange')
    plt.title(f'Log-Transformed Distribution: {col}')
    plt.xlabel(f'log1p({col})')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
columns_to_scale = df_cleaned.select_dtypes(include=['int64', 'float64']).columns.tolist()
columns_to_scale = [col for col in columns_to_scale if col not in ['id', 'Date', 'Postal Code']]

scaler = StandardScaler()

df_cleaned[columns_to_scale] = scaler.fit_transform(df_cleaned[columns_to_scale])

print("DataFrame after Standard Scaling:")
display(df_cleaned.head())

print("\nDescriptive statistics of scaled numerical features:")
display(df_cleaned[columns_to_scale].describe())

In [ ]:
df_cleaned.to_csv("Cleaned_Tabular.csv")

In [ ]:
df_cleaned.shape

# TEXT DATASET

In [ ]:
df2=pd.read_csv("text.csv")
df2.head(10)

In [ ]:
df2.isnull().sum()

In [ ]:
df2.shape

In [ ]:
df2.info()

In [ ]:
import re
import string

# 1. Lowercase Conversion
df2['Message'] = df2['Message'].str.lower()
print("Messages after lowercase conversion (first 5):")
display(df2['Message'].head())


In [ ]:
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

df2['Message'] = df2['Message'].apply(remove_html_tags)
print("Messages after HTML tag removal (first 5):")
display(df2['Message'].head())


In [ ]:
def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

df2['Message'] = df2['Message'].apply(remove_urls)
print("Messages after URL removal (first 5):")
display(df2['Message'].head())


In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df2['Message'] = df2['Message'].apply(remove_punctuation)
print("Messages after punctuation removal (first 5):")
display(df2['Message'].head())


In [ ]:
def remove_special_characters(text):

    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

df2['Message'] = df2['Message'].apply(remove_special_characters)
print("Messages after special character removal (first 5):")
display(df2['Message'].head())


In [ ]:
initial_message_count = df2.shape[0]
df2.drop_duplicates(subset=['Message'], inplace=True)

duplicates_removed = initial_message_count - df2.shape[0]
print(f"Number of duplicate messages removed: {duplicates_removed}")
print(f"DataFrame shape after duplicate removal: {df2.shape}")
print("Messages after duplicate removal (first 5):")
display(df2.head())


In [ ]:
! pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data (if not already downloaded)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4')
try:
    nltk.data.find('tokenizers/punkt') # Added to download the punkt tokenizer
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab') # Added to download the punkt_tab resource
except LookupError:
    nltk.download('punkt_tab')

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

print("NLTK setup complete.")

In [ ]:
df2['Tokens'] = df2['Message'].apply(nltk.word_tokenize)
print("Messages after tokenization (first 5 tokens):")
display(df2['Tokens'].head())


In [ ]:
df2['Tokens_NoStopwords'] = df2['Tokens'].apply(lambda x: [word for word in x if word not in stop_words])
print("Messages after stopword removal (first 5 tokens):")
display(df2['Tokens_NoStopwords'].head())


In [ ]:
df2['Processed_Message'] = df2['Tokens_NoStopwords'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

# Join the processed tokens back into a string for easier viewing/further processing if needed
df2['Processed_Message_Str'] = df2['Processed_Message'].apply(lambda x: ' '.join(x))

print("Messages after lemmatization and joined (first 5):")
display(df2[['Message', 'Processed_Message_Str']].head())


In [ ]:
df2.head()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all processed messages into a single string
all_words = ' '.join(df2['Processed_Message_Str'])

# Generate a word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_words)

# Display the generated image:
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Processed Messages')
plt.show()


In [ ]:
from nltk.probability import FreqDist

# Flatten the list of tokenized words
all_tokens = [word for sublist in df2['Processed_Message'] for word in sublist]

# Get frequency distribution
freq_dist = FreqDist(all_tokens)

# Get the top 20 most frequent words
top_20_words = freq_dist.most_common(20)

# Create a DataFrame for plotting
df_top_words = pd.DataFrame(top_20_words, columns=['Word', 'Frequency'])

# Plot the top 20 words
plt.figure(figsize=(12, 7))
sns.barplot(x='Frequency', y='Word', data=df_top_words, palette='viridis')
plt.title('Top 20 Most Frequent Words')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.show()


In [ ]:
# Calculate the length of each processed message (number of tokens)
df2['Message_Length'] = df2['Processed_Message'].apply(len)

# Plot the distribution of message lengths
plt.figure(figsize=(8, 4))
sns.histplot(df2['Message_Length'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Processed Message Lengths (in tokens)')
plt.xlabel('Message Length (Tokens)')
plt.ylabel('Number of Messages')
plt.show()


In [ ]:
# Flatten the list of all processed tokens (already done for FreqDist, but good to re-use or re-define for clarity)
all_processed_tokens = [word for sublist in df2['Processed_Message'] for word in sublist]

# Total number of words in the corpus
total_words = len(all_processed_tokens)

# Vocabulary size (number of unique words)
vocabulary_size = len(set(all_processed_tokens))

print(f"Total number of words in the cleaned corpus: {total_words}")
print(f"Vocabulary size (unique words): {vocabulary_size}")

In [ ]:
df2.to_csv("cleaned_text.csv", index=False)

In [ ]:
df2.shape

# IMAGE DATASET

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yashdogra/cats-and-dogs")

print("Path to dataset files:", path)

In [ ]:
import os

image_dir = os.path.join(path, 'images') # Corrected path based on directory listing

# List all image files
image_files = []
for root, _, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(root, file))

print(f"Found {len(image_files)} image files.")

# Display first few image paths
print("First 5 image paths:")
for i in range(min(5, len(image_files))):
    print(image_files[i])

In [ ]:
from PIL import Image
import numpy as np

valid_image_files = []
corrupted_images_count = 0

for img_path in image_files:
    try:
        with Image.open(img_path) as img:
            img.verify() # Verify if the image is valid
        valid_image_files.append(img_path)
    except (IOError, SyntaxError) as e:
        corrupted_images_count += 1
        # print(f"Corrupted image removed: {img_path} - {e}") # Uncomment to see which images are corrupted

print(f"Total valid images: {len(valid_image_files)}")
print(f"Total corrupted images removed: {corrupted_images_count}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

image_dimensions = []

for img_path in valid_image_files:
    try:
        with Image.open(img_path) as img:
            image_dimensions.append(img.size) # img.size returns (width, height)
    except Exception as e:
        print(f"Could not read dimensions for {img_path}: {e}")

# Convert to DataFrame for easier analysis
dimensions_df = pd.DataFrame(image_dimensions, columns=['Width', 'Height'])

print("Descriptive statistics of image dimensions:")
display(dimensions_df.describe())

# Plot distributions of Width and Height
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(dimensions_df['Width'], bins=50, kde=True)
plt.title('Distribution of Image Widths')
plt.xlabel('Width (pixels)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(dimensions_df['Height'], bins=50, kde=True)
plt.title('Distribution of Image Heights')
plt.xlabel('Height (pixels)')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
target_size = (224, 224)
resized_images = []

for img_path in valid_image_files:
    try:
        with Image.open(img_path) as img:

            img = img.convert('RGB')
            img = img.resize(target_size)
            resized_images.append(np.array(img))
    except Exception as e:
        print(f"Could not resize image {img_path}: {e}")

print(f"Successfully resized {len(resized_images)} images to {target_size}.")

# Display the shape of the first resized image to confirm
if resized_images:
    print(f"Shape of the first resized image: {resized_images[0].shape}")

#### 5. Normalize Pixel Values

Pixel values in images typically range from 0 to 255. Normalizing these values to a range of 0 to 1 helps in stabilizing and speeding up the training process of neural networks. I will convert the list of resized image arrays into a single NumPy array and then normalize them.

In [ ]:
normalized_images = np.array(resized_images).astype('float32') / 255.0

print(f"Shape of the normalized images array: {normalized_images.shape}")
print(f"Min pixel value after normalization: {normalized_images.min()}")
print(f"Max pixel value after normalization: {normalized_images.max()}")

In [ ]:
!pip install imagehash

import imagehash
from PIL import Image
from tqdm import tqdm

image_hashes = {}

pil_images_for_hashing = []
for img_array in resized_images:
    pil_images_for_hashing.append(Image.fromarray(img_array.astype('uint8')))

duplicate_indices = set()

print("Calculating image hashes and identifying duplicates...")
for i, img_pil in enumerate(tqdm(pil_images_for_hashing)):
    current_hash = imagehash.phash(img_pil)

    if current_hash in image_hashes:
        duplicate_indices.add(i)

    else:
        image_hashes[current_hash] = i # Store the index of the first occurrence

print(f"Found {len(duplicate_indices)} duplicate images.")

# Filter out duplicate images from normalized_images and valid_image_files
final_normalized_images = []
final_image_paths = []

for i in range(len(normalized_images)):
    if i not in duplicate_indices:
        final_normalized_images.append(normalized_images[i])
        final_image_paths.append(valid_image_files[i])

final_normalized_images = np.array(final_normalized_images)

print(f"Number of unique images after removing duplicates: {len(final_normalized_images)}")
print(f"Shape of the final unique normalized images array: {final_normalized_images.shape}")

In [ ]:
import matplotlib.pyplot as plt
import random

# Display a few sample processed images
plt.figure(figsize=(12, 8))
num_samples_to_show = 5

for i in range(num_samples_to_show):
    idx = random.randint(0, len(final_normalized_images) - 1)

    plt.subplot(1, num_samples_to_show, i + 1)
    plt.imshow(final_normalized_images[idx])
    plt.title(f"Image {idx}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Plot pixel intensity histogram for a sample image
if len(final_normalized_images) > 0:
    sample_image_idx = random.randint(0, len(final_normalized_images) - 1)
    sample_image = final_normalized_images[sample_image_idx]

    plt.figure(figsize=(8, 5))
    sns.histplot(sample_image.flatten(), bins=50, color='skyblue', kde=True)
    plt.title(f'Pixel Intensity Histogram for Sample Image {sample_image_idx}')
    plt.xlabel('Pixel Intensity (0-1)')
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No processed images to display histogram for.")

In [ ]:
np.save('cleaned_normalized_images.npy', final_normalized_images)

with open('cleaned_image_paths.txt', 'w') as f:
    for item in final_image_paths:
        f.write(f"{item}\n")

print("Cleaned and normalized images saved to 'cleaned_normalized_images.npy'.")
print("Corresponding image paths saved to 'cleaned_image_paths.txt'.")

In [ ]:
file_path = "cleaned_normalized_images.npy"

size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024**2)
size_gb = size_bytes / (1024**3)

print(f"Size: {size_bytes} bytes")
print(f"Size: {size_mb:.2f} MB")
print(f"Size: {size_gb:.2f} GB")